# Train MatGPT Base Models on a Colab T4

This notebook runs the base-pretraining pipeline for the course validation models:

- `MatGPT-Mini 8M` on TinyStories
- `MatGPT-Tiny 59M` on BabyLM-2026-Strict

Before running: choose **Runtime -> Change runtime type -> T4 GPU**. Active data stays on fast, ephemeral `/content`; prepared artifacts, run evidence, and checkpoints persist in Google Drive.

## 1. Choose one stage

Run `prepare`, then `smoke`, then `pilot`, and then `evaluate`. Reaching pilot step 306 is not approval: review the persisted evidence with Codex before manually selecting `full`. Select `evaluate` again after the full run.

In [ ]:
# @title Run settings
MODEL = "mini_8m"  # @param ["mini_8m", "tiny_59m"]
RUN_STAGE = "prepare"  # @param ["prepare", "smoke", "pilot", "full", "evaluate"]
SMOKE_MAX_STEPS = 20
RESUME_CHECK_STEPS = 5
PILOT_STOP_STEP = 306

# If you publish this project to GitHub, paste the repo URL here.
# If left blank, the notebook expects PROJECT_DIR to already contain this repo.
REPO_URL = ""  # @param {type:"string"}
PROJECT_DIR = "/content/train-llm-from-scratch"  # @param {type:"string"}

# W&B is optional but recommended for a serious run because it preserves curves
# even if the Colab tab disconnects.
ENABLE_WANDB = True  # @param {type:"boolean"}
WANDB_PROJECT = "matgpt-t4-base"  # @param {type:"string"}
WANDB_ENTITY = ""  # @param {type:"string"}
assert MODEL in {"mini_8m", "tiny_59m"}
assert RUN_STAGE in {"prepare", "smoke", "pilot", "full", "evaluate"}

## 2. Mount Google Drive

Colab VMs disappear. Drive stores durable checkpoints, metrics, preflight and benchmark JSON, plus synchronized copies of prepared artifacts.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Locate or clone the project

This cell keeps the project checkout on local `/content`. If the GitHub repo is private, add a Colab secret named `GITHUB_TOKEN` with repo read access. If `REPO_URL` is blank, `PROJECT_DIR` must already contain the repository.

In [ ]:
import os
import subprocess
from pathlib import Path
from urllib.parse import urlparse, urlunparse

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_colab_secret(name: str):
    """Read a Colab secret if it exists, otherwise return None."""
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

def github_clone_url(repo_url: str, token: str | None) -> str:
    """Attach a GitHub token for private HTTPS clones without changing REPO_URL."""
    if not token or not repo_url.startswith("https://github.com/"):
        return repo_url
    parsed = urlparse(repo_url)
    return urlunparse(parsed._replace(netloc=f"x-access-token:{token}@{parsed.netloc}"))

def git_run(args, label: str, token: str | None = None):
    """Run git and show stdout/stderr. This avoids opaque CalledProcessError traces."""
    result = subprocess.run(args, text=True, capture_output=True)
    stdout = result.stdout.replace(token, "<GITHUB_TOKEN>") if token else result.stdout
    stderr = result.stderr.replace(token, "<GITHUB_TOKEN>") if token else result.stderr
    if result.returncode != 0:
        raise RuntimeError(
            f"{label} with exit code {result.returncode}.\n\n"
            f"stdout:\n{stdout}\n\n"
            f"stderr:\n{stderr}\n\n"
            "If this is a private GitHub repo, add a Colab secret named GITHUB_TOKEN "
            "with repo read access, then rerun this cell."
        )
    if stdout:
        print(stdout)
    if stderr:
        print(stderr)
    return result

project_dir = Path(PROJECT_DIR)
github_token = get_colab_secret("GITHUB_TOKEN")

if project_dir.exists() and (project_dir / ".git").exists():
    print("Existing git checkout found. Pulling latest changes.")
    git_pull = git_run(["git", "-C", str(project_dir), "pull", "--ff-only"], "git pull", github_token)
elif REPO_URL:
    if project_dir.exists() and any(project_dir.iterdir()):
        raise RuntimeError(
            f"PROJECT_DIR exists but is not a git checkout: {project_dir}\n"
            "Choose a different PROJECT_DIR, delete the partial folder, or copy the repo there manually."
        )
    project_dir.parent.mkdir(parents=True, exist_ok=True)
    clone_url = github_clone_url(REPO_URL, github_token)
    git_run(["git", "clone", clone_url, str(project_dir)], "Git clone failed", github_token)
elif not project_dir.exists():
    raise RuntimeError(
        "REPO_URL is blank and PROJECT_DIR does not exist. Set REPO_URL to the GitHub repo, "
        "or place the project checkout at PROJECT_DIR first."
    )

os.chdir(project_dir)
print("Current directory:", Path.cwd())
assert Path("scripts/pretrain.py").exists(), "PROJECT_DIR must point at the MatGPT repository root."

## 4. Install dependencies

This installs the local package, dataset tools, W&B, and optional Colab helpers such as bitsandbytes.

In [ ]:
%pip install -q -e ".[test,colab]" huggingface_hub wandb

## 5. Authenticate Hugging Face and W&B

Recommended Colab secrets:

- `HF_TOKEN`: Hugging Face token for dataset access
- `WANDB_API_KEY`: Weights & Biases API key

If a secret is missing, the notebook falls back to the normal interactive login prompt.

In [ ]:
from huggingface_hub import login as hf_login
import wandb

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_colab_secret(name: str):
    """Read a Colab secret if it exists, otherwise return None."""
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

hf_token = get_colab_secret("HF_TOKEN")
if hf_token:
    hf_login(token=hf_token, add_to_git_credential=False)
else:
    hf_login(add_to_git_credential=False)

if ENABLE_WANDB:
    wandb_key = get_colab_secret("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key)
    else:
        wandb.login()
else:
    print("W&B disabled for this run.")

## 6. Gate storage and the GPU

Every stage prints local and Drive capacity before expensive work. Preparation requires at least 20 GiB free on `/content`; training stages additionally require CUDA on an NVIDIA T4.

In [ ]:
!nvidia-smi

import shutil
import torch

LOCAL_MIN_FREE_GIB = 20
local_usage = shutil.disk_usage("/content")
drive_usage = shutil.disk_usage("/content/drive/MyDrive")
print("/content disk usage:", local_usage)
print("Drive disk usage:", drive_usage)
assert local_usage.free >= LOCAL_MIN_FREE_GIB * 1024**3, (
    f"Need at least {LOCAL_MIN_FREE_GIB} GiB free on /content; "
    f"observed {local_usage.free / 1024**3:.2f} GiB."
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unavailable"
print("gpu:", gpu_name)
if RUN_STAGE in {"smoke", "pilot", "full"}:
    assert torch.cuda.is_available(), f"{RUN_STAGE} requires CUDA."
    assert "T4" in gpu_name, f"{RUN_STAGE} requires an NVIDIA T4; observed {gpu_name!r}."

## 7. Build the fixed-path Colab config

All stages use the unchanged full training schedule. Normalized data, tokenizer files, and shards work from a stable local path and are restored from or synchronized to Drive; run evidence writes directly to Drive.

In [ ]:
import json
import shutil
import yaml
from pathlib import Path

base_config_path = Path("configs/matgpt_mini_8m.yaml") if MODEL == "mini_8m" else Path("configs/matgpt_tiny_59m.yaml")
cfg = yaml.safe_load(base_config_path.read_text())

LOCAL_ROOT = Path("/content/matgpt_work") / cfg["run"]["name"]
DRIVE_ROOT = Path("/content/drive/MyDrive/matgpt_artifacts") / cfg["run"]["name"]
cfg["run"]["output_dir"] = str(DRIVE_ROOT / "run")
cfg["dataset"]["normalized_dir"] = str(LOCAL_ROOT / "normalized")
cfg["tokenizer"]["output_dir"] = str(LOCAL_ROOT / "tokenizer")
cfg["sharding"]["output_dir"] = str(LOCAL_ROOT / "shards")

cfg["tracking"]["wandb"]["enabled"] = bool(ENABLE_WANDB)
cfg["tracking"]["wandb"]["project"] = WANDB_PROJECT
cfg["tracking"]["wandb"]["entity"] = WANDB_ENTITY or None

PERSISTED_ARTIFACT_DIRS = ("normalized", "tokenizer", "shards")


def restore_artifacts_from_drive():
    restored = []
    for name in PERSISTED_ARTIFACT_DIRS:
        local_path = LOCAL_ROOT / name
        drive_path = DRIVE_ROOT / name
        if not local_path.exists() and drive_path.exists():
            local_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(drive_path, local_path)
            restored.append(name)
    print("Restored from Drive:", restored or "nothing")


def sync_artifacts_to_drive():
    synchronized = []
    for name in PERSISTED_ARTIFACT_DIRS:
        local_path = LOCAL_ROOT / name
        drive_path = DRIVE_ROOT / name
        if local_path.exists():
            drive_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(local_path, drive_path, dirs_exist_ok=True)
            synchronized.append(name)
    print("Synchronized to Drive:", synchronized or "nothing")


LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
restore_artifacts_from_drive()

colab_config_dir = LOCAL_ROOT / "config"
colab_config_dir.mkdir(parents=True, exist_ok=True)
colab_config_path = colab_config_dir / f"{MODEL}.yaml"
colab_config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

print("Local working root:", LOCAL_ROOT)
print("Persistent Drive root:", DRIVE_ROOT)
print("Persistent run directory:", cfg["run"]["output_dir"])
print("Local normalized data:", cfg["dataset"]["normalized_dir"])
print("Local tokenizer:", cfg["tokenizer"]["output_dir"])
print("Local shards:", cfg["sharding"]["output_dir"])
print("Wrote config:", colab_config_path)
print(json.dumps({
    "model": MODEL,
    "run_stage": RUN_STAGE,
    "max_tokens": cfg["training"]["max_tokens"],
    "wandb_enabled": cfg["tracking"]["wandb"]["enabled"],
}, indent=2))

## 8. Prepare once, then synchronize

The `prepare` stage runs normalization, tokenizer training, and sharding only when each complete output set is absent. Successful local artifacts are then synchronized to Drive. Other stages only use the fixed local copy restored above.

In [ ]:
import shlex
import subprocess


def run_command(command):
    """Run a command, expose its evidence, and fail with full diagnostics."""
    printable = " ".join(shlex.quote(str(part)) for part in command)
    print("\n$", printable)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {printable}\n\n"
            f"stdout:\n{result.stdout}\n\n"
            f"stderr:\n{result.stderr}"
        )
    return result


validation_split = (
    cfg["dataset"].get("validation_split")
    or cfg["dataset"].get("generated_validation_split")
)
preparation_steps = [
    (
        "normalized dataset",
        [
            Path(cfg["dataset"]["normalized_dir"]) / "manifest.json",
            Path(cfg["dataset"]["normalized_dir"]) / f"{cfg['dataset']['train_split']}.jsonl",
            Path(cfg["dataset"]["normalized_dir"]) / f"{validation_split}.jsonl",
        ],
        ["python", "scripts/prepare_dataset.py", "--config", str(colab_config_path)],
    ),
    (
        "tokenizer",
        [
            Path(cfg["tokenizer"]["output_dir"]) / "tokenizer.json",
            Path(cfg["tokenizer"]["output_dir"]) / "special_tokens.json",
        ],
        ["python", "scripts/train_tokenizer.py", "--config", str(colab_config_path)],
    ),
    (
        "token shards",
        [
            Path(cfg["sharding"]["output_dir"]) / f"{cfg['dataset']['train_split']}_metadata.json",
            Path(cfg["sharding"]["output_dir"]) / f"{validation_split}_metadata.json",
        ],
        ["python", "scripts/tokenize_and_shard.py", "--config", str(colab_config_path)],
    ),
]

if RUN_STAGE == "prepare":
    for label, expected_paths, command in preparation_steps:
        if all(path.is_file() for path in expected_paths):
            print(f"Skipping {label}; expected outputs exist.")
        else:
            run_command(command)
            missing = [str(path) for path in expected_paths if not path.is_file()]
            assert not missing, f"{label} completed without expected outputs: {missing}"
    sync_artifacts_to_drive()
else:
    print(f"Preparation commands skipped for stage {RUN_STAGE!r}.")

## 9. Gate training with preflight and benchmark evidence

Every training stage reruns strict T4 preflight and a five-step batch benchmark. Review the persisted `preflight.json` and `benchmark.json`; training is blocked unless the configured batch has finite loss and pre-clip gradient norm, positive throughput, and memory fraction below 0.90.

In [ ]:
import math

TRAINING_STAGES = {"smoke", "pilot", "full"}
batch_sizes = "8,16,24,32" if MODEL == "mini_8m" else "2,4,6,8"

if RUN_STAGE in TRAINING_STAGES:
    run_command([
        "python", "scripts/preflight_t4.py",
        "--config", str(colab_config_path),
        "--require-t4",
        "--min-free-disk-gb", "20",
    ])
    preflight_path = Path(cfg["run"]["output_dir"]) / "preflight.json"
    preflight_report = json.loads(preflight_path.read_text(encoding="utf-8"))
    print("Persisted preflight evidence:", preflight_path)
    print(json.dumps(preflight_report, indent=2, sort_keys=True))
    assert preflight_report["status"] == "pass", "Preflight gate did not pass."

    benchmark = run_command([
        "python", "scripts/benchmark_t4.py",
        "--config", str(colab_config_path),
        "--batch-sizes", batch_sizes,
        "--steps", "5",
    ])
    benchmark_path = Path(cfg["run"]["output_dir"]) / "benchmark.json"
    benchmark_path.parent.mkdir(parents=True, exist_ok=True)
    benchmark_path.write_text(benchmark.stdout, encoding="utf-8")
    benchmark_report = json.loads(benchmark.stdout)
    selected_batch = next(
        result
        for result in benchmark_report["results"]
        if result["batch_size"] == cfg["training"]["micro_batch_size"]
    )
    print("Persisted benchmark evidence:", benchmark_path)
    print(json.dumps({
        "batch_size": selected_batch["batch_size"],
        "loss": selected_batch.get("loss"),
        "grad_norm": selected_batch.get("grad_norm"),
        "memory_fraction": selected_batch.get("memory_fraction"),
        "tokens_per_second": selected_batch.get("tokens_per_second"),
    }, indent=2, sort_keys=True))
    if selected_batch["status"] != "ok":
        raise RuntimeError(f"Configured batch benchmark failed: {selected_batch}")
    for field in ("loss", "grad_norm", "tokens_per_second", "memory_fraction"):
        if not math.isfinite(float(selected_batch[field])):
            raise FloatingPointError(f"Benchmark {field} is not finite: {selected_batch[field]}")
    if selected_batch["tokens_per_second"] <= 0:
        raise RuntimeError("Benchmark throughput must be positive.")
    if selected_batch["memory_fraction"] >= 0.90:
        raise RuntimeError(f"Benchmark memory fraction is too high: {selected_batch['memory_fraction']:.3f}")
else:
    print(f"Preflight and benchmark are training-only gates; skipped for {RUN_STAGE!r}.")

## 10. Run the selected training stage

`--max-steps` always means additional successful optimizer updates; it never rewrites `max_tokens` or the full learning-rate schedule (6,104 steps for the 200M-token Mini run). Smoke performs 20 updates plus a separate five-update resume check. Pilot stops at global step 306. Reaching 306 is not approval: the user and Codex must review the evidence before the operator manually selects `full`.

In [ ]:
base_train_cmd = ["python", "scripts/pretrain.py", "--config", str(colab_config_path)]
latest = Path(cfg["run"]["output_dir"]) / "checkpoints" / "latest.pt"

if RUN_STAGE == "smoke":
    assert not latest.exists(), "Smoke must begin from a new run directory."
    run_command(base_train_cmd + ["--max-steps", str(SMOKE_MAX_STEPS)])
    smoke_step = int(torch.load(latest, map_location="cpu", weights_only=False)["state"]["global_step"])
    assert smoke_step == SMOKE_MAX_STEPS, f"Smoke stopped at unexpected step {smoke_step}."
    run_command(base_train_cmd + ["--resume-from", str(latest), "--max-steps", str(RESUME_CHECK_STEPS)])
    resumed_step = int(torch.load(latest, map_location="cpu", weights_only=False)["state"]["global_step"])
    assert resumed_step == SMOKE_MAX_STEPS + RESUME_CHECK_STEPS
    print(f"Smoke and resume check passed at global step {resumed_step}.")
elif RUN_STAGE == "pilot":
    assert latest.exists(), "Run the smoke and resume check first."
    current_step = int(torch.load(latest, map_location="cpu", weights_only=False)["state"]["global_step"])
    remaining = PILOT_STOP_STEP - current_step
    assert remaining > 0, f"Pilot already reached: global_step={current_step}"
    run_command(base_train_cmd + ["--resume-from", str(latest), "--max-steps", str(remaining)])
    pilot_step = int(torch.load(latest, map_location="cpu", weights_only=False)["state"]["global_step"])
    assert pilot_step == PILOT_STOP_STEP, f"Pilot stopped at unexpected step {pilot_step}."
    print("Pilot reached step 306. Select evaluate; do not select full yet.")
elif RUN_STAGE == "full":
    assert latest.exists(), "Run and approve the pilot first."
    current_step = int(torch.load(latest, map_location="cpu", weights_only=False)["state"]["global_step"])
    assert current_step >= PILOT_STOP_STEP, "Pilot gate has not been reached."
    run_command(base_train_cmd + ["--resume-from", str(latest)])
else:
    print(f"No training command runs during stage {RUN_STAGE!r}.")

## 11. Evaluate every available review checkpoint

The `evaluate` stage evaluates `latest.pt` and `best.pt` when present, then writes the durable run summary. Run this after pilot and again after full training.

In [ ]:
run_dir = Path(cfg["run"]["output_dir"])
checkpoint_dir = run_dir / "checkpoints"
best = checkpoint_dir / "best.pt"
latest = checkpoint_dir / "latest.pt"

if RUN_STAGE == "evaluate":
    evaluation_checkpoints = [path for path in (latest, best) if path.exists()]
    assert evaluation_checkpoints, "No checkpoint found. Run training first."
    for checkpoint in evaluation_checkpoints:
        run_command([
            "python", "scripts/evaluate.py",
            "--config", str(colab_config_path),
            "--checkpoint", str(checkpoint),
        ])
    run_command(["python", "scripts/summarize_run.py", "--run-dir", str(run_dir)])
else:
    print(f"Evaluation commands skipped for stage {RUN_STAGE!r}.")

## 12. Review the persisted gate evidence

This review is intentionally manual. It displays the summary, last 30 metric rows, losses, skipped updates, memory behavior, benchmark fraction, evaluation files, and latest sample. The calculations never promote the notebook to `full`.

In [ ]:
import math

import pandas as pd
from IPython.display import Markdown, display

if RUN_STAGE == "evaluate":
    summary_path = run_dir / "run_summary.md"
    assert summary_path.is_file(), f"Missing run summary: {summary_path}"
    display(Markdown(summary_path.read_text(encoding="utf-8")))

    metrics_path = run_dir / "metrics.csv"
    assert metrics_path.is_file(), f"Missing metrics: {metrics_path}"
    metrics = pd.read_csv(metrics_path)
    display(metrics.tail(30))
    for column in ["train_loss", "val_loss"]:
        finite_rows = metrics.dropna(subset=[column]).copy()
        finite_rows[column] = pd.to_numeric(finite_rows[column], errors="coerce")
        finite_rows = finite_rows[finite_rows[column].map(math.isfinite)]
        finite_rows.plot(x="tokens_processed", y=column, title=column)

    skipped_columns = [
        column for column in (
            "global_step",
            "optimizer_step_skipped",
            "optimizer_steps_skipped_total",
            "consecutive_optimizer_steps_skipped",
        )
        if column in metrics.columns
    ]
    display(metrics.loc[metrics["event"] == "train", skipped_columns].tail(30))

    finite_training_fields = [
        "train_loss", "tokens_per_second", "peak_memory_mb",
        "optimizer_steps_skipped_total",
    ]
    for field in finite_training_fields:
        metrics[field] = pd.to_numeric(metrics[field], errors="coerce")
    train_rows = metrics.dropna(subset=["train_loss"]).copy()
    train_rows = train_rows[
        train_rows[finite_training_fields].apply(
            lambda row: all(math.isfinite(float(value)) for value in row), axis=1
        )
    ]
    window = max(5, math.ceil(len(train_rows) * 0.20))
    assert len(train_rows) >= window * 2, "Not enough logged pilot rows for early/late comparison."
    early = train_rows.head(window)
    late = train_rows.tail(window)
    assert float(early["tokens_per_second"].median()) > 0
    pilot_gate = {
        "early_loss_median": float(early["train_loss"].median()),
        "late_loss_median": float(late["train_loss"].median()),
        "throughput_ratio": float(late["tokens_per_second"].median() / early["tokens_per_second"].median()),
        "maximum_peak_memory_mb": float(train_rows["peak_memory_mb"].max()),
        "skipped_updates_total": int(train_rows["optimizer_steps_skipped_total"].max()),
    }
    pilot_gate["loss_improved"] = pilot_gate["late_loss_median"] < pilot_gate["early_loss_median"]
    pilot_gate["throughput_stable"] = pilot_gate["throughput_ratio"] >= 0.80

    val_rows = metrics.dropna(subset=["val_loss"]).copy()
    val_rows["val_loss"] = pd.to_numeric(val_rows["val_loss"], errors="coerce")
    val_rows = val_rows[val_rows["val_loss"].map(math.isfinite)]
    assert len(val_rows) >= 2, "Need at least two finite validation rows for comparison."
    pilot_gate["first_val_loss"] = float(val_rows.iloc[0]["val_loss"])
    pilot_gate["last_val_loss"] = float(val_rows.iloc[-1]["val_loss"])
    pilot_gate["validation_improved"] = pilot_gate["last_val_loss"] < pilot_gate["first_val_loss"]

    benchmark_path = run_dir / "benchmark.json"
    benchmark_report = json.loads(benchmark_path.read_text(encoding="utf-8"))
    selected_batch = next(
        result for result in benchmark_report["results"]
        if result["batch_size"] == cfg["training"]["micro_batch_size"]
    )
    pilot_gate["benchmark_memory_fraction"] = float(selected_batch["memory_fraction"])
    print(json.dumps(pilot_gate, indent=2, sort_keys=True))

    peak_memory_values = train_rows["peak_memory_mb"].tolist()
    if len(peak_memory_values) > 1 and all(
        current > previous for previous, current in zip(peak_memory_values, peak_memory_values[1:])
    ):
        print("WARNING: every logged peak-memory value is strictly greater than its predecessor.")

    evaluation_files = sorted((run_dir / "evaluation").glob("*.json"))
    print("Evaluation files:", [str(path) for path in evaluation_files])
    sample_files = sorted((run_dir / "samples").glob("samples_*.json"))
    if sample_files:
        print("Latest sample file:", sample_files[-1])
        print(sample_files[-1].read_text(encoding="utf-8")[:4000])
    else:
        print("No periodic sample file exists yet.")
else:
    print("Select evaluate after pilot to review promotion evidence.")

## 13. Generate from the checkpoint

This uses the base model as a text completer. It is not a chatbot yet; SFT and DPO come later.

In [ ]:
if RUN_STAGE == "evaluate":
    checkpoint = best if best.exists() else latest
    prompt = "Once upon a time" if MODEL == "mini_8m" else "A token is"
    run_command([
        "python", "scripts/chat.py",
        "--config", str(colab_config_path),
        "--checkpoint", str(checkpoint),
        "--prompt", prompt,
        "--max-new-tokens", "120",
    ])
else:
    print("Generation is available in the evaluate stage.")